# Day 20 — Debugging PyTorch Masterclass

## 1. Learning Objectives
- Develop a systematic approach to debugging PyTorch errors.
- Diagnose and fix Shape and Dimensionality errors.
- Diagnose and fix Device and Dtype mismatches.
- Troubleshoot silent logical errors (like NaN loss and exploding gradients).

## 2. Prerequisites
- Completion of Days 1 through 19.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## 3. Concept Explanation
You spend 20% of your time writing PyTorch code and 80% of your time debugging it.
The 3 absolute most common errors you will encounter are:
1. **Shape Mismatch**: Trying to matrix multiply `[32, 10]` by `[64, 20]`.
2. **Device Mismatch**: "Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!"
3. **Dtype Mismatch**: Trying to pass a `Long` (Integer) tensor into a Linear layer that expects `Float`.

Let's break them.

## Bug 1: The Shape Mismatch
Look at the code below. Run it in your head. Why will it crash?

In [ ]:
class BadShapeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(64, 1) # <--- BUG IS HERE
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# dummy_x = torch.randn(5, 10)
# model = BadShapeModel()
# model(dummy_x) # Will throw RuntimeError: mat1 and mat2 shapes cannot be multiplied

**Fix**: The output of `fc1` is 32 features. The input of `fc2` expects 64 features. They must match! Change to `nn.Linear(32, 1)`.

## Bug 2: The Dtype Mismatch
Neural network weights are initialized as 32-bit floats (`torch.float32`). If you feed them integers, they crash.

In [ ]:
linear = nn.Linear(2, 1)
int_data = torch.tensor([[1, 2], [3, 4]]) # Defaults to torch.int64

# linear(int_data) # Will throw RuntimeError: expected scalar type Float but found Long

**Fix**: Cast your data to float before passing to the model: `linear(int_data.float())`.

## Bug 3: The Target Shape Error (Broadcasting Disaster)
This is a silent bug. It won't crash, but your loss won't go down properly.

In [ ]:
pred = torch.randn(32, 1) # Model outputs shape [Batch, 1]
target = torch.randn(32)  # Labels are shape [Batch]

criterion = nn.MSELoss()
loss = criterion(pred, target)
print("Nonsense Loss due to broadcasting:", loss.item())

**Explanation**: Because `pred` is `[32, 1]` and `target` is `[32]`, PyTorch broadcasts `target` to `[32, 32]`, calculates a massive 32x32 MSE matrix, and averages it. The math is valid, but completely wrong for your training task.
**Fix**: Ensure shapes match exactly! Use `target = target.unsqueeze(1)` or `target.view(-1, 1)` to make it `[32, 1]`.

## Bug 4: Exploding Gradients / NaN Loss
You are training a model, and suddenly your loss prints `NaN` (Not a Number).

**Causes & Fixes**:
1. **Learning Rate too high**: The optimizer took a massive step into infinity. Fix: Lower `lr`.
2. **Unnormalized Data**: A feature has a value of 50,000, causing massive outputs. Fix: Standardize inputs.
3. **Bad Loss Function usage**: Passing probabilities to `BCEWithLogitsLoss` instead of raw logits, or passing negative numbers to a `log` function.

## 11. Practice Exercise 1: The Ultimate Debugging Test
The code below has 3 bugs. It is trying to do binary classification. Fix the code so it runs without errors.

In [ ]:
import torch.nn.functional as F

class BrokenNet(nn.Module):
    def __init__(self):
        # Bug 1 here?
        self.fc = nn.Linear(5, 1)
        
    def forward(self, x):
        return self.fc(x)

net = BrokenNet()
optimizer = optim.Adam(net.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

data = torch.randn(10, 5)
labels = torch.randint(0, 2, (10,)) # Bug 2 here?

for i in range(3):
    preds = net(data)
    # loss = criterion(preds, labels) # Uncomment. Bug 3 here?
    # loss.backward()
    # optimizer.step()
    # optimizer.zero_grad()

In [ ]:
# SOLUTION
# Bug 1: Missing super().__init__() in the BrokenNet class.
# Bug 2 & 3: labels is shape [10] (int64). preds is shape [10, 1] (float32). 
# Fix: labels = torch.randint(0, 2, (10, 1)).float()
# Then criterion(preds, labels) will work perfectly.

## 19. Day Summary
When your PyTorch code crashes, follow this checklist:
1. `print(tensor.shape)` everywhere. Ensure `mat1` inner matches `mat2` inner.
2. `print(tensor.dtype)`. Models want floats. `CrossEntropyLoss` targets want Longs (Ints). `BCE` targets want floats.
3. `print(tensor.device)`. Ensure model and data are on the same device.
4. Check for silent broadcasting errors in your Loss calculation (`[Batch, 1]` vs `[Batch]`).